# Synthetic Data Benchmark

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd

from utils_fit import benchmark_distribution_fits, benchmark_bezierv_mse_methods


## MLE

In [2]:
df = benchmark_distribution_fits(
    k=200,
    n_samples=1000,
    n_bezierv=10,
    seed=42,
    output_csv="data/benchmark_results.csv",
    verbose=True,
)
df.head(12)

Benchmarking: 100%|██████████| 200/200 [00:31<00:00,  6.27instance/s]


Results saved to 'data/benchmark_results.csv'  (600 rows, 200 instances).


,Dist,Method,Time,NegLogLik
0,uniform,bezierv_mle,0.003953,2845.679234
1,uniform,generalized_beta,0.049968,2846.168083
2,uniform,johnson,0.027199,3019.260080
3,uniform,bezierv_mle,0.002669,2970.294519
4,uniform,generalized_beta,0.048492,2973.002095
5,uniform,johnson,0.044530,3134.535649
6,triangular,bezierv_mle,0.013822,2132.953214
7,triangular,generalized_beta,0.031537,2133.540246
8,triangular,johnson,0.039408,2165.072885
9,lognormal,bezierv_mle,0.066795,2909.277031


### Tables

In [3]:
summary = (
    df.groupby("Method")[["NegLogLik", "Time"]]
    .agg(["mean", "median", "std"])
    .round(4)
)
summary

NegLogLik                         Time                
                       mean     median       std    mean  median     std
Method                                                                  
bezierv_mle       2074.5812  2278.5442  787.6567  0.0436  0.0280  0.0383
generalized_beta  2153.5032  2351.0492  828.7400  0.0312  0.0306  0.0135
johnson           2149.1470  2354.8190  824.0306  0.0362  0.0410  0.0108

In [4]:

METHOD_ORDER = ["bezierv_mle", "generalized_beta", "johnson"]
METHOD_LABELS = {
    "bezierv_mle":      "Bézier MLE",
    "generalized_beta": "Gen. Beta",
    "johnson":          "Johnson SU",
}

summary_dist = (
    df.groupby(["Dist", "Method"])[["NegLogLik", "Time"]]
    .agg(["mean", "median", "std"])
    .round(4)
)

# Pivot to wide format: rows = Dist, columns = (Method, metric)
pivot_nll_mean = (
    df.groupby(["Dist", "Method"])["NegLogLik"]
    .mean()
    .unstack("Method")
    [["bezierv_mle", "generalized_beta", "johnson"]]
    .rename(columns={k: f"{v} (mean)" for k, v in METHOD_LABELS.items()})
    .round(1)
)

pivot_nll_std = (
    df.groupby(["Dist", "Method"])["NegLogLik"]
    .std()
    .unstack("Method")
    [["bezierv_mle", "generalized_beta", "johnson"]]
    .rename(columns={k: f"{v} (std)" for k, v in METHOD_LABELS.items()})
    .round(1)
)

pivot_nll = pivot_nll_mean.join(pivot_nll_std)
# Interleave mean and std columns
cols = [col for pair in zip(pivot_nll_mean.columns, pivot_nll_std.columns) for col in pair]
pivot_nll = pivot_nll[cols]

pivot_nll.index = pivot_nll.index.str.replace("_", " ").str.title()
pivot_nll.index.name = "Distribution"

# Add average row
avg_row = pivot_nll.mean().round(1).rename("Average")
pivot_nll = pd.concat([pivot_nll, avg_row.to_frame().T])
pivot_nll.index.name = "Distribution"

print("Mean ± Std NLL by distribution and method")
pivot_nll


Mean ± Std NLL by distribution and method


Method,Bézier MLE (mean),Bézier MLE (std),Gen. Beta (mean),Gen. Beta (std),Johnson SU (mean),Johnson SU (std)
Distribution,,,,,,
Beta,1527.5,899.7,1521.1,901.9,1592.2,916.8
Bimodal Gaussian,2613.5,258.4,2835.4,267.4,2800.3,288.7
Exponential,2334.8,645.4,2368.2,680.2,2364.6,652.3
Gamma,2840.3,583.8,2834.6,582.6,2834.6,582.1
Log Logistic,1579.0,741.7,1703.1,774.8,1566.5,740.5
Lognormal,1478.3,922.4,1505.2,946.4,1468.3,919.7
Normal,2231.4,482.3,2221.2,482.0,2221.2,481.8
Triangular,1818.2,623.1,1821.4,622.9,1848.8,624.8
Trimodal Gaussian,2438.5,285.6,2850.6,198.0,2738.4,336.0


In [5]:

# Pivot to wide format for Time: rows = Dist, columns = (Method, metric)
pivot_time_mean = (
    df.groupby(["Dist", "Method"])["Time"]
    .mean()
    .mul(1000)
    .unstack("Method")
    [["bezierv_mle", "generalized_beta", "johnson"]]
    .rename(columns={k: f"{v} (mean)" for k, v in METHOD_LABELS.items()})
    .round(1)
)

pivot_time_std = (
    df.groupby(["Dist", "Method"])["Time"]
    .std()
    .mul(1000)
    .unstack("Method")
    [["bezierv_mle", "generalized_beta", "johnson"]]
    .rename(columns={k: f"{v} (std)" for k, v in METHOD_LABELS.items()})
    .round(1)
)

pivot_time = pivot_time_mean.join(pivot_time_std)
# Interleave mean and std columns
cols_t = [col for pair in zip(pivot_time_mean.columns, pivot_time_std.columns) for col in pair]
pivot_time = pivot_time[cols_t]

pivot_time.index = pivot_time.index.str.replace("_", " ").str.title()
pivot_time.index.name = "Distribution"

# Add average row
avg_row_t = pivot_time.mean().round(1).rename("Average")
pivot_time = pd.concat([pivot_time, avg_row_t.to_frame().T])
pivot_time.index.name = "Distribution"

print("Mean ± Std Fitting Time (ms) by distribution and method")
pivot_time.style.format("{:.1f}")


Mean ± Std Fitting Time (ms) by distribution and method


Method,Bézier MLE (mean),Bézier MLE (std),Gen. Beta (mean),Gen. Beta (std),Johnson SU (mean),Johnson SU (std)
Distribution,,,,,,
Beta,17.9,7.6,20.0,8.8,42.5,4.3
Bimodal Gaussian,91.2,29.3,31.0,10.0,34.0,12.4
Exponential,20.6,4.3,42.3,4.6,41.0,1.0
Gamma,24.4,7.9,33.9,11.1,41.7,5.1
Log Logistic,83.2,32.8,45.2,3.5,17.2,6.7
Lognormal,54.4,36.6,45.1,5.7,36.7,8.8
Normal,46.5,20.4,27.9,14.5,33.1,12.1
Triangular,13.2,2.0,16.1,5.4,43.4,5.4
Trimodal Gaussian,88.1,39.4,35.8,13.1,28.8,11.1


## MSE

In [6]:
df = benchmark_bezierv_mse_methods(
    k=200,
    n_samples=1000,
    n_bezierv=10,
    seed=42,
    output_csv="data/benchmark_results.csv",
    verbose=True
)

df.head(3)

MSE benchmark: 100%|██████████| 200/200 [1:00:07<00:00, 18.04s/instance]


Results saved to 'data/benchmark_results.csv'  (600 rows, 200 instances).
No fitting failures.


,Dist,Method,Time,MSE,Failed
0,uniform,nonlinear,2.281648,0.000008,False
1,uniform,projgrad,0.059001,0.000013,False
2,uniform,neldermead,11.287757,0.000019,False


### Tables

In [7]:
METHOD_ORDER_MSE = ["nonlinear", "projgrad", "neldermead"]
METHOD_LABELS_MSE = {
    "nonlinear":  "Nonlinear",
    "projgrad":   "Proj. Grad.",
    "neldermead": "Nelder-Mead",
}

fails = (
    df[df["Failed"]]
    .groupby("Method")
    .size()
    .reindex(METHOD_ORDER_MSE, fill_value=0)
    .rename(index=METHOD_LABELS_MSE)
    .rename("Failures")
    .to_frame()
)
fails["Total"] = df.groupby("Method").size().reindex(METHOD_ORDER_MSE).values
fails["Failure Rate (%)"] = (fails["Failures"] / fails["Total"] * 100).round(2)
fails


,Failures,Total,Failure Rate (%)
Method,,,
Nonlinear,0,200,0.0
Proj. Grad.,0,200,0.0
Nelder-Mead,0,200,0.0


In [10]:
pivot_mse_mean = (
    df.groupby(["Dist", "Method"])["MSE"]
    .mean()
    .unstack("Method")
    [METHOD_ORDER_MSE]
    .rename(columns={k: f"{v} (mean)" for k, v in METHOD_LABELS_MSE.items()})
)

pivot_mse_std = (
    df.groupby(["Dist", "Method"])["MSE"]
    .std()
    .unstack("Method")
    [METHOD_ORDER_MSE]
    .rename(columns={k: f"{v} (std)" for k, v in METHOD_LABELS_MSE.items()})
)

pivot_mse = pivot_mse_mean.join(pivot_mse_std)
cols_mse = [col for pair in zip(pivot_mse_mean.columns, pivot_mse_std.columns) for col in pair]
pivot_mse = pivot_mse[cols_mse]

pivot_mse.index = pivot_mse.index.str.replace("_", " ").str.title()
pivot_mse.index.name = "Distribution"

avg_row_mse = pivot_mse.mean().rename("Average")
pivot_mse = pd.concat([pivot_mse, avg_row_mse.to_frame().T])
pivot_mse.index.name = "Distribution"

print("Mean ± Std MSE by distribution and method")
pivot_mse.style.format("{:.1e}")


Mean ± Std MSE by distribution and method


Method,Nonlinear (mean),Nonlinear (std),Proj. Grad. (mean),Proj. Grad. (std),Nelder-Mead (mean),Nelder-Mead (std)
Distribution,,,,,,
Beta,1.1e-05,3.8e-06,1.9e-05,7.3e-06,3.9e-04,4.5e-04
Bimodal Gaussian,3.9e-05,6.6e-05,2.4e-04,3.1e-04,1.3e-03,1.8e-03
Exponential,1.1e-05,3.4e-06,2.1e-05,8.6e-06,1.0e-04,4.3e-05
Gamma,1.3e-05,3.9e-06,2.3e-05,8.1e-06,1.8e-04,8.4e-05
Log Logistic,1.3e-05,4.8e-06,8.0e-03,1.7e-02,2.1e-03,1.9e-03
Lognormal,1.3e-05,3.6e-06,4.8e-04,1.3e-03,8.7e-04,7.8e-04
Normal,1.2e-05,2.7e-06,2.0e-05,8.0e-06,8.8e-04,6.1e-04
Triangular,1.3e-05,3.7e-06,1.7e-05,4.2e-06,8.9e-05,9.0e-05
Trimodal Gaussian,2.5e-04,2.5e-04,2.3e-03,7.0e-03,2.8e-03,2.3e-03


In [9]:

# --- Time table ---
pivot_time_mse_mean = (
    df.groupby(["Dist", "Method"])["Time"]
    .mean()
    .mul(1000)
    .unstack("Method")
    [METHOD_ORDER_MSE]
    .rename(columns={k: f"{v} (mean)" for k, v in METHOD_LABELS_MSE.items()})
    .round(1)
)

pivot_time_mse_std = (
    df.groupby(["Dist", "Method"])["Time"]
    .std()
    .mul(1000)
    .unstack("Method")
    [METHOD_ORDER_MSE]
    .rename(columns={k: f"{v} (std)" for k, v in METHOD_LABELS_MSE.items()})
    .round(1)
)

pivot_time_mse = pivot_time_mse_mean.join(pivot_time_mse_std)
cols_t_mse = [col for pair in zip(pivot_time_mse_mean.columns, pivot_time_mse_std.columns) for col in pair]
pivot_time_mse = pivot_time_mse[cols_t_mse]

pivot_time_mse.index = pivot_time_mse.index.str.replace("_", " ").str.title()
pivot_time_mse.index.name = "Distribution"

avg_row_t_mse = pivot_time_mse.mean().round(1).rename("Average")
pivot_time_mse = pd.concat([pivot_time_mse, avg_row_t_mse.to_frame().T])
pivot_time_mse.index.name = "Distribution"

print("Mean ± Std Fitting Time (ms) by distribution and method")
pivot_time_mse.style.format("{:.1f}")


Mean ± Std Fitting Time (ms) by distribution and method


Method,Nonlinear (mean),Nonlinear (std),Proj. Grad. (mean),Proj. Grad. (std),Nelder-Mead (mean),Nelder-Mead (std)
Distribution,,,,,,
Beta,3042.8,2874.1,60.9,10.3,14659.1,3646.3
Bimodal Gaussian,2379.9,653.3,69.2,49.1,15928.2,7360.7
Exponential,2469.8,615.0,56.1,3.3,15324.1,761.2
Gamma,2502.8,502.4,59.5,10.0,16623.3,5137.1
Log Logistic,2386.1,561.2,67.4,33.8,18148.6,5214.2
Lognormal,2732.1,803.6,76.7,44.0,16558.4,1769.1
Normal,2254.3,577.7,67.3,36.4,15565.6,7411.3
Triangular,2328.4,311.9,60.6,8.6,14860.3,3917.9
Trimodal Gaussian,2303.9,449.6,60.8,7.6,14922.1,1266.9
